# Unidade 6 - Pratica: Tuning de Hiperparametros e Explainable AI (Roteiro)

**Disciplina:** Machine Learning

Nesta pratica, voce vai ajustar um modelo de deteccao de fraude em transacoes financeiras (base `transactions.csv`, estilo PaySim) usando tres estrategias de tuning - Grid Search, Random Search e Optuna - e depois explicar o modelo final com SHAP.

**Objetivos**
- Comparar Grid Search, Random Search e Optuna em custo computacional e qualidade do resultado.
- Usar `average_precision` como metrica de tuning em um problema desbalanceado (retomando a Unidade 5).
- Interpretar um modelo de floresta aleatoria com SHAP (importancia global e explicacoes locais).
- Usar os insights do SHAP para orientar uma nova rodada de engenharia de atributos.


In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import randint
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay, PrecisionRecallDisplay, average_precision_score,
    f1_score, precision_score, recall_score
)
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split

import optuna
from optuna.visualization.matplotlib import plot_optimization_history
import shap

SEED = 42
sns.set_theme(style='whitegrid')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# TODO: carregue 'transactions.csv' em df e explore:
# - dimensoes e head()
# - contagem de transacoes por 'type'
# - taxa de fraude geral (df['isFraud'].mean())
# - taxa de fraude por tipo de transacao (groupby('type')['isFraud'].mean())

df = pd.read_csv('transactions.csv')


## 1. Recorte e amostragem dos dados
A fraude so ocorre nos tipos `CASH_OUT` e `TRANSFER`. Mantenha apenas essas transacoes, o que ja reduz bastante o ruido. Mesmo assim, a base ainda teria fraude muito rara (menos de 1%). Para que o treinamento e o tuning rodem em poucos segundos durante a aula, mantenha todas as fraudes e sorteie uma amostra de transacoes legitimas, criando uma base menor porem ainda desbalanceada (em torno de 4-5% de fraude).

Note que **nao** vamos criar ainda as variaveis de erro de saldo (`errorBalanceOrig`/`errorBalanceDest`). Elas aparecerao mais adiante, guiadas pela explicabilidade do modelo.

### Passos
1. Filtre `df` para `type` em `['CASH_OUT', 'TRANSFER']`.
2. Crie a coluna `isTransfer` (1 se `type == 'TRANSFER'`, 0 caso contrario).
3. Monte `df_sample` com todas as fraudes + uma amostra de 6000 transacoes legitimas (`random_state=SEED`).
4. Defina `feature_cols` com as 7 colunas base (sem os erros de saldo) e separe `X`/`y`.
5. Faca `train_test_split` com `test_size=0.25`, `random_state=SEED` e `stratify=y`.

### Perguntas
- Por que faz sentido restringir a base aos tipos `CASH_OUT` e `TRANSFER`?
- Por que amostrar as transacoes legitimas em vez de usar todas?

In [ ]:
# TODO: filtre df para type in ['CASH_OUT', 'TRANSFER'] e crie a coluna isTransfer.
# df_txn = ...

# TODO: monte df_sample com todas as fraudes + amostra de 6000 legitimas (random_state=SEED).
# frauds = ...
# non_frauds = ...
# df_sample = ...

# TODO: defina feature_cols (7 colunas base, sem colunas de erro de saldo).
# feature_cols = ['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig',
#                 'oldbalanceDest', 'newbalanceDest', 'isTransfer']
# X = ...
# y = ...

# TODO: X_train, X_test, y_train, y_test = train_test_split(...)

print('Prepare df_sample, X, y e a divisao treino/teste.')


## 2. Modelo baseline
Antes de qualquer tuning, treine uma `RandomForestClassifier` com os hiperparametros padrao. Esse resultado sera seu ponto de comparacao: qualquer tecnica de tuning precisa justificar o tempo investido mostrando ganho sobre esse baseline.

Como a fraude e a classe minoritaria, acompanhe F1, precisao, recall e **Average Precision** (area sob a curva precisao-recall), a mesma logica de metricas discutida na Unidade 5.

### Perguntas
- Qual metrica voce esperaria que fosse mais sensivel ao desbalanceamento: acuracia ou Average Precision?

In [ ]:
# TODO: treine um RandomForestClassifier(random_state=SEED, n_jobs=-1) padrao em X_train/y_train.
# Meca o tempo de treino com time.perf_counter().
# Calcule F1, precisao, recall e Average Precision no conjunto de teste.

# baseline = ...
# pred_baseline = ...
# proba_baseline = ...

print('Treine e avalie o modelo baseline.')


## 3. Grid Search: busca exaustiva
O Grid Search testa **todas as combinacoes** de uma grade de valores pre-definida. E simples de entender e garante cobrir todo o espaco declarado, mas o custo cresce multiplicativamente: cada novo hiperparametro multiplica o numero de combinacoes.

Use `scoring='average_precision'` porque essa e a metrica mais adequada para o desbalanceamento da base. Atencao: **deixe `n_jobs=-1` apenas na busca (`GridSearchCV`) e `n_jobs=1` no estimador** - colocar `n_jobs=-1` nos dois ao mesmo tempo sobrecarrega os nucleos do processador e deixa a busca mais lenta, nao mais rapida.

### Passos
1. Defina `param_grid` com `n_estimators`, `max_depth` e `min_samples_leaf`.
2. Calcule quantas combinacoes existem na grade.
3. Rode `GridSearchCV` com `cv=5` medindo o tempo total.

### Perguntas
- Quantas combinacoes existem na grade abaixo? Quantos ajustes de modelo isso significa considerando `cv=5`?
- O que aconteceria com o tempo total se adicionassemos mais um hiperparametro com 3 valores?

In [ ]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [4, 8, None],
    'min_samples_leaf': [1, 5]
}
# TODO: calcule n_combinacoes multiplicando o tamanho de cada lista em param_grid.
# n_combinacoes = ...

# TODO: crie e treine um GridSearchCV(RandomForestClassifier(random_state=SEED, n_jobs=1), ...)
# medindo o tempo com time.perf_counter(). Use scoring='average_precision' e cv=5.

# grid_search = ...
# grid_time = ...

print('Rode o Grid Search e imprima best_params_, best_score_ e o tempo gasto.')


## 4. Random Search: amostragem aleatoria
O Random Search sorteia combinacoes de um espaco de distribuicoes (continuas ou discretas) por um numero fixo de iteracoes (`n_iter`). Ele nao garante testar tudo, mas em geral explora o espaco de forma mais eficiente que o Grid Search quando ha muitos hiperparametros, porque nao desperdica tentativas repetindo valores pouco informativos de uma dimensao enquanto outra dimensao fica fixa.

### Passos
1. Defina `param_dist` usando `scipy.stats.randint` para os hiperparametros numericos e uma lista para `max_features`.
2. Rode `RandomizedSearchCV` com `n_iter=15`, `cv=5`, `scoring='average_precision'`.

### Perguntas
- Compare `n_iter=15` do Random Search com as `n_combinacoes` do Grid Search. Qual buscou em menos combinacoes?
- O resultado (`best_score_`) do Random Search superou o do Grid Search? Isso sempre acontece?

In [ ]:
# TODO: defina param_dist com randint para n_estimators, max_depth, min_samples_leaf
#       e uma lista para max_features (ex.: ['sqrt', 'log2', None]).
# param_dist = {...}

# TODO: crie e treine um RandomizedSearchCV(..., n_iter=15, scoring='average_precision',
#       cv=5, n_jobs=-1, random_state=SEED) medindo o tempo gasto.
# random_search = ...
# random_time = ...

print('Rode o Random Search e imprima best_params_, best_score_ e o tempo gasto.')


## 5. Optuna: busca guiada (Bayesiana/TPE)
Grid e Random Search escolhem a proxima tentativa sem olhar para o resultado das tentativas anteriores. O Optuna usa um **sampler adaptativo** (por padrao, o TPE - Tree-structured Parzen Estimator), que aprende com os resultados ja observados para escolher regioes mais promissoras do espaco de busca a cada nova tentativa (`trial`).

Cada `trial` treina o modelo com uma combinacao de hiperparametros e retorna uma metrica (aqui, a media do `average_precision` em validacao cruzada de 3 folds - usamos 3 em vez de 5 para manter o orcamento de tempo da aula). O `study.optimize` repete esse processo por `n_trials` tentativas, refinando a busca a cada rodada.

### Passos
1. Escreva uma funcao `objective(trial)` que sugere hiperparametros com `trial.suggest_int`/`trial.suggest_categorical`, treina o modelo com `cross_val_score` (`cv=3`) e retorna a media do `average_precision`.
2. Crie um `study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))`.
3. Rode `study.optimize(objective, n_trials=15)`.
4. Plote o historico de otimizacao com `plot_optimization_history(study)`.

### Perguntas
- Observando o grafico de historico, em que momento o Optuna encontrou sua melhor tentativa?
- O Optuna avaliou menos, mais ou o mesmo numero de combinacoes que o Random Search? O resultado justifica a diferenca de tempo?

In [ ]:
# TODO: escreva a funcao objective(trial) descrita acima.
# Dica: dentro dela, use RandomForestClassifier(random_state=SEED, n_jobs=1, **params)
# e cross_val_score(model, X_train, y_train, cv=3, scoring='average_precision', n_jobs=-1).mean()

# def objective(trial):
#     ...

# TODO: crie o study e rode study.optimize(objective, n_trials=15), medindo o tempo.
# study = ...
# optuna_time = ...

# TODO: plote o historico de otimizacao com plot_optimization_history(study).

print('Rode o Optuna e imprima best_params, best_value e o tempo gasto.')


## 6. Comparando as tres estrategias
Nenhuma das tres tecnicas e universalmente "melhor". Grid Search e exaustivo e previsivel, mas caro. Random Search e simples e escala melhor com muitos hiperparametros. Optuna costuma encontrar bons resultados com menos tentativas porque aprende com o historico, mas tem um pouco mais de complexidade de configuracao (e overhead de orquestrar cada `trial`).

Monte uma tabela com tempo gasto e melhor pontuacao de validacao cruzada de cada metodo, e visualize com graficos de barra.

In [ ]:
# TODO: monte um DataFrame 'comparacao' com colunas Metodo, Tempo (s),
#       Avg. Precision (CV) e Configuracoes avaliadas para os tres metodos.
# comparacao = pd.DataFrame([...])
# display(comparacao)

# TODO: crie dois graficos de barra (tempo e qualidade) lado a lado com plt.subplots.

print('Compare tempo e qualidade dos tres metodos de tuning.')


## 7. Modelo final tunado
Use os hiperparametros encontrados pelo Optuna (nossa busca mais direcionada) para treinar o modelo final no conjunto de treino completo e avaliar no conjunto de teste, que nao participou de nenhuma etapa de tuning.

### Passos
1. Treine `RandomForestClassifier(random_state=SEED, n_jobs=-1, **study.best_params)` em `X_train`/`y_train`.
2. Calcule F1, precisao, recall e Average Precision no teste, comparando com o baseline.
3. Plote a matriz de confusao e a curva precisao-recall.

In [ ]:
# TODO: treine final_model com os melhores hiperparametros do Optuna.
# final_model = ...

# TODO: calcule pred_final e proba_final e imprima as metricas, comparando com o baseline.

# TODO: plote ConfusionMatrixDisplay.from_predictions e PrecisionRecallDisplay.from_predictions
#       lado a lado.

print('Avalie o modelo final tunado no conjunto de teste.')


## 8. Explicabilidade com SHAP
O tuning melhora metricas, mas nao explica **por que** o modelo decide o que decide. Em fraude, isso importa: um analista precisa justificar por que uma transacao foi bloqueada, e a equipe de risco precisa confiar que o modelo aprendeu padroes razoaveis (e nao um artefato dos dados).

Use `shap.TreeExplainer`, adequado para modelos baseados em arvore como a Random Forest. Para um classificador binario, o SHAP retorna valores por classe; use a fatia da classe positiva (fraude = 1): `explicacao.values[:, :, 1]`.

### Passos
1. Crie `explainer = shap.TreeExplainer(final_model)` e `explicacao = explainer(X_test)`.
2. Plote `shap.summary_plot(..., plot_type='bar')` (importancia media) e o `summary_plot` padrao (dispersao).

### Perguntas
- Quais atributos mais empurram a previsao em direcao a fraude?
- O grafico de dispersao mostra, para o atributo mais importante, se valores altos ou baixos aumentam o risco previsto. O que isso sugere sobre o padrao de fraude nesta base?

In [ ]:
# TODO: crie o explainer e a explicacao para X_test.
# explainer = shap.TreeExplainer(final_model)
# explicacao = explainer(X_test)
# shap_fraude = explicacao.values[:, :, 1]

# TODO: plote shap.summary_plot(shap_fraude, X_test, plot_type='bar', show=False) + plt.show()
# TODO: plote shap.summary_plot(shap_fraude, X_test, show=False) + plt.show()

print('Gere os graficos de importancia global do SHAP.')


## 9. Explicacoes locais: por que *este* caso foi (ou nao) sinalizado?
A importancia global mostra o padrao médio do modelo. Para auditar uma decisao especifica, use um grafico *waterfall*, que mostra como cada atributo empurra a previsao daquela transacao para cima ou para baixo a partir de um valor base.

### Passos
1. Encontre o indice de uma fraude corretamente identificada (`y_test == 1` e `pred_final == 1`).
2. Plote `shap.plots.waterfall(explicacao[idx, :, 1])` para esse caso.
3. Se houver algum falso negativo (`y_test == 1` e `pred_final == 0`), repita o processo para esse caso.

### Perguntas
- No caso corretamente identificado, quais atributos mais contribuiram para a previsao de fraude?
- Se houve um falso negativo, o que os valores de SHAP sugerem sobre por que o modelo nao sinalizou essa transacao?

In [ ]:
# TODO: encontre idx_fraude_detectada com np.where((y_test.values == 1) & (pred_final == 1))[0][0]
# TODO: plote shap.plots.waterfall(explicacao[idx_fraude_detectada, :, 1], show=False) + plt.show()

# TODO: procure um falso negativo com np.where((y_test.values == 1) & (pred_final == 0))[0]
#       e, se existir, plote o waterfall dele tambem.

print('Explique um caso corretamente identificado e, se houver, um falso negativo.')


## 10. Engenharia de atributos guiada pelo SHAP
Os graficos globais deste modelo tendem a apontar `oldbalanceOrg`, `newbalanceOrig` e as colunas de saldo do destinatario como os atributos mais influentes. Isso sugere uma hipotese: o que importa pode nao ser o saldo em si, mas a **consistencia contabil** da transacao - o quanto o saldo mudou bateu com o valor transferido.

Vamos testar essa hipotese criando duas novas variaveis:
- `errorBalanceOrig = newbalanceOrig + amount - oldbalanceOrg` (deveria ser 0 numa transacao consistente)
- `errorBalanceDest = oldbalanceDest + amount - newbalanceDest` (idem, do lado do destinatario)

Reaproveite os **mesmos hiperparametros** encontrados pelo Optuna - o objetivo aqui e isolar o efeito da nova informacao, nao fazer um novo tuning.

### Passos
1. Crie as duas colunas de erro de saldo em `df_sample`.
2. Monte `feature_cols_v2` incluindo as novas colunas e refaca o `train_test_split`.
3. Treine `modelo_v2` com os mesmos `best_params` e compare as metricas com o modelo da secao 7.

In [ ]:
# TODO: crie df_sample['errorBalanceOrig'] e df_sample['errorBalanceDest'] conforme descrito.

# TODO: monte feature_cols_v2 = feature_cols + ['errorBalanceOrig', 'errorBalanceDest']
#       e refaca o train_test_split (mesmos parametros da secao 1).

# TODO: treine modelo_v2 = RandomForestClassifier(random_state=SEED, n_jobs=-1, **best_params)

# TODO: monte uma tabela comparando F1, precisao, recall e Average Precision
#       do modelo original (secao 7) com o modelo_v2.

print('Compare o desempenho antes e depois da engenharia de atributos guiada pelo SHAP.')


### Perguntas
- O ganho de desempenho confirma a hipotese levantada a partir do SHAP?
- Esse tipo de ciclo (modelo -> explicacao -> nova hipotese -> novo atributo) e uma das razoes praticas para investir em explicabilidade, alem da conformidade e da confianca. Que outros atributos voce imaginaria testar em um cenario real de deteccao de fraude?

## Conclusao
Registre: como Grid Search, Random Search e Optuna se compararam em tempo e qualidade nesta pratica; o que o SHAP revelou sobre o comportamento do modelo; e como a explicacao do modelo motivou uma melhoria concreta de desempenho. Em um projeto real de deteccao de fraude, tuning e explicabilidade sao complementares: um melhora a metrica, o outro melhora a confianca e o entendimento sobre o que essa metrica realmente representa.